In [ ]:
import pandas as pd
import os
import pickle
from wsi_stats import WSIStatsCache

# Path to WSIs (root dir)
ROOT_DIR = "//regsj.intern/appl/Deep_Visual_Proteomics"

# Path to cache file
CACHE_FILE = "wsi_cache_christine.pkl"

# Load/reload WSI stats cache
wsi_cache = WSIStatsCache(ROOT_DIR, CACHE_FILE)
df_wsi = wsi_cache.main(reload=True)

In [ ]:
# Path to pathology metadata
df_path = "D:\DATA\initial_cleaning.xlsx"
df_pathology = pd.read_excel(df_path)

In [ ]:
# Check for WSI with no associated data
missing_data = df_wsi[
    (df_wsi["file_size"].isna() | (df_wsi["file_size"] == 0)) |
    (df_wsi["data_folder_size"].isna() | (df_wsi["data_folder_size"] == 0))
]

print("WSI with missing data: ", missing_data)

In [ ]:
# Drop rows with no associated data
rows_to_drop = missing_data[missing_data.any(axis=1)].index
df_no_missing = df_wsi.drop(index=rows_to_drop)

print(f"Original rows: {len(df_wsi)}, After dropping: {len(df_no_missing)}")

In [ ]:
print(df_pathology["rekvnr"].head())
print(df_no_missing["rekvnr"].head())

In [ ]:
df_no_missing["rekvnr"] = pd.to_numeric(df_no_missing["rekvnr"], errors="coerce").astype("Int64")

In [ ]:
rekvnr_pathology = set(df_pathology["rekvnr"])
rekvnr_wsi = set(df_no_missing["rekvnr"])

overlap = rekvnr_pathology & rekvnr_wsi
only_in_pathology = rekvnr_pathology - rekvnr_wsi
only_in_wsi = rekvnr_wsi - rekvnr_pathology

print(f"rekvnr in both: {len(overlap)}")
print(f"rekvnr only in pathology: {len(only_in_pathology)}")
print(f"rekvnr only in WSI: {len(only_in_wsi)}")

In [ ]:
count_in_wsi = df_no_missing["rekvnr"].isin(overlap).sum()
print(f"Number of rows in WSI with overlapping rekvnr: {count_in_wsi}")

counts_per_rekvnr = df_no_missing[df_no_missing["rekvnr"].isin(overlap)]["rekvnr"].value_counts()
print(counts_per_rekvnr)

In [ ]:
# Subset pathology DataFrame to only overlapping rekvnr
df_overlap = df_pathology[df_pathology["rekvnr"].isin(overlap)].copy()

# Count how many WSI files exist for each rekvnr
wsi_counts = df_no_missing["rekvnr"].value_counts()

# Add the count as a new column in the pathology subset
df_overlap["wsi count"] = df_overlap["rekvnr"].map(wsi_counts).fillna(0).astype(int)

print(df_overlap.head())

In [ ]:
# Create a mapping from rekvnr to list of filenames
filenames_per_rekvnr = df_no_missing.groupby("rekvnr").apply(
    lambda g: list(g.index),
    include_groups=False
)

# Add the list of filenames as a new column in df_overlap
df_overlap["wsi filenames"] = df_overlap["rekvnr"].map(filenames_per_rekvnr)

print(df_overlap[["rekvnr", "wsi count", "wsi filenames"]].head())

In [ ]:
# Save to Excel
output_file = "D:\DATA\overlapping_rekvnr.xlsx"
df_overlap.to_excel(output_file, index=False)

print(f"Saved DataFrame to {output_file}")